# Run the arivis retinal-layer model in Python

This notebook runs the arivis Cloud model (`model.czann` or `model.czmodel`) on TIFF
images without ZEISS software, and optionally compares the result with a segmentation
exported from arivis/ZEN.

> **Note:** the segmentation masks published with the paper were generated in arivis Cloud.
> To reproduce them exactly, run the model in arivis Cloud, arivis Pro or ZEN. This notebook
> is an alternative for users without ZEISS software; its output can differ slightly from
> arivis, mainly at boundaries between layers.

**Steps**
1. Set the paths in the **Settings** cell below.
2. Run all cells (*Run → Run All Cells*).
3. Validation runs only if `VALIDATION_PATH` is set.

**Requirements:** see `environment.yml` in this folder
(`conda env create -f environment.yml`, `conda activate zf-arivis-model`, then `jupyter lab`).

**Model classes and label values** (same as ZEN):
1 background, 2 GCL, 3 INL, 4 IPL, 5 LENS, 6 ON, 7 ONL (= photoreceptor layer, PRL)

## Settings
Relative paths are relative to this notebook's folder. On Windows, keep the `r` in front of paths, e.g. `r"D:\data\fish1.tif"`.

In [ ]:
# ---- Model: path to model.czann (new format) or model.czmodel (legacy format) ----
MODEL_PATH = ""

# ---- Input: a single .tif/.tiff file, or a folder of them ----
# Each file can be a 2D slice or a 3D stack (slices along the first axis).
INPUT_PATH = ""

# ---- Output folder ----
OUTPUT_DIR = ""

# ---- Classes to also save as 0/255 masks (empty list [] to skip) ----
# Available: "background", "GCL", "INL", "IPL", "LENS", "ON", "ONL"
SAVE_CLASSES = []

# ---- Optional validation against an arivis/ZEN segmentation ----
# None to skip. Otherwise a file (when INPUT_PATH is a single file) or a folder of
# reference images. In a folder, references are matched to inputs by file name; extra
# endings such as "_finalprediction_ome" (reference) or "_raw" (input) are ignored.
VALIDATION_PATH = None

# What the reference images contain:
#   None   -> the arivis export with one 0/255 channel per class (*_finalprediction.ome.tiff),
#             or a label image with one value per class (e.g. 1-7); detected automatically
#   "ONL"  -> a binary mask of a single class (any non-zero pixel = that class)
VALIDATION_CLASS = None

# First label value in the reference label image (ZEN uses 1 = background).
# Set to 0 if your reference labels run 0-6.
REFERENCE_FIRST_LABEL = 1

# Number of tiles predicted at once (lower this if you run out of memory)
BATCH_SIZE = 16

## Functions
No changes needed here.

In [ ]:
import json
import re
import warnings
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path

import numpy as np
import tifffile
import onnxruntime as ort
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

TIFF_SUFFIXES = {".tif", ".tiff"}
LABEL_COLOURS = ["#000000", "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231", "#911eb4"]


def load_model(model_path):
    """Read the ONNX model and metadata from a .czann or .czmodel file."""
    model_path = Path(model_path)
    with zipfile.ZipFile(model_path) as z:
        names = z.namelist()
        model_bytes = z.read(next(n for n in names if n.endswith(".model")))
        json_files = [n for n in names if n.endswith(".json")]
        xml_files = [n for n in names if n.endswith(".xml")]
        if json_files:                                   # new format (.czann)
            meta = json.loads(z.read(json_files[0]))
            tile = tuple(meta["InputShape"][:2])
            overlap = tuple(meta["MinOverlap"])
            classes = meta["Classes"]
        elif xml_files:                                  # legacy format (.czmodel)
            root = ET.fromstring(z.read(xml_files[0]))
            border = int(root.findtext("BorderSize"))
            overlap = (border, border)
            items = sorted(root.find("TrainingClasses"), key=lambda i: int(i.get("LabelValue")))
            classes = [i.get("Name") for i in items]
            tile = None
        else:
            raise ValueError(f"No metadata (.json or .xml) found in {model_path}")

    session = ort.InferenceSession(model_bytes, providers=ort.get_available_providers())
    inp = session.get_inputs()[0]
    info = {
        "tile": tile or tuple(inp.shape[1:3]),
        "overlap": overlap,
        "classes": classes,
        "input_name": inp.name,
        "scale_to_unit": expects_unit_range(model_bytes, model_path),
    }
    return session, info


def expects_unit_range(model_bytes, model_path):
    """True if the model expects intensities scaled to 0-1 (.czann), False for raw 16-bit (.czmodel).

    Read from the model's built-in normalisation; falls back to the file extension.
    """
    try:
        import onnx
        from onnx import numpy_helper
        graph = onnx.load_from_string(model_bytes).graph
        values = {t.name: numpy_helper.to_array(t) for t in graph.initializer}
        for node in graph.node:
            if node.op_type == "Constant":
                values[node.output[0]] = numpy_helper.to_array(node.attribute[0].t)
        for node in graph.node:
            if node.op_type == "Add" and "shift" in node.name.lower():
                shift = [values[i] for i in node.input if i in values]
                if shift:
                    return abs(float(np.ravel(shift[0])[0])) < 10
    except ImportError:
        pass
    return Path(model_path).suffix.lower() == ".czann"


def tile_starts(length, tile, overlap):
    """Tile start positions spread evenly so the tiles exactly cover `length`,
    with neighbouring tiles overlapping by at least `overlap` pixels."""
    if length <= tile:
        return [0]
    n = int(np.ceil((length - overlap) / (tile - overlap)))
    return [int(round(i * (length - tile) / (n - 1))) for i in range(n)]


def blend_weights(th, tw):
    """Linear weights, highest at the tile centre and lowest at its edges."""
    wy = np.minimum(np.arange(th) + 1, th - np.arange(th))
    wx = np.minimum(np.arange(tw) + 1, tw - np.arange(tw))
    return np.outer(wy, wx).astype(np.float32)[..., None]


def predict_slice(image, session, info, batch_size=16):
    """Segment one 2D image. Returns class probabilities, shape (rows, cols, n_classes).

    The image is covered by an evenly spaced grid of tiles that overlap by at least the
    model's overlap (24 px), without padding the image edges. Where tiles overlap, their
    predictions are averaged with linear weights that favour each tile's centre.
    Images smaller than one tile are mirror-padded up to the tile size.
    """
    th, tw = info["tile"]
    oh, ow = info["overlap"]
    rows, cols = image.shape

    x = image.astype(np.float32)
    if info["scale_to_unit"]:
        x = x / 65535.0
    pad_r, pad_c = max(0, th - rows), max(0, tw - cols)
    if pad_r or pad_c:
        x = np.pad(x, ((0, pad_r), (0, pad_c)), mode="reflect")
    H, W = x.shape

    coords = [(y, c) for y in tile_starts(H, th, oh) for c in tile_starts(W, tw, ow)]
    weights = blend_weights(th, tw)
    acc = np.zeros((H, W, len(info["classes"])), np.float32)
    wsum = np.zeros((H, W, 1), np.float32)
    for b in range(0, len(coords), batch_size):
        batch = coords[b:b + batch_size]
        tiles = np.stack([x[y:y + th, c:c + tw] for y, c in batch])[..., None]
        probs = session.run(None, {info["input_name"]: tiles})[0]
        for (y, c), p in zip(batch, probs):
            acc[y:y + th, c:c + tw] += p * weights
            wsum[y:y + th, c:c + tw] += weights
    return (acc / wsum)[:rows, :cols]


def check_image(image, path):
    if image.ndim not in (2, 3):
        raise ValueError(f"{path}: expected a 2D image or 3D stack, got shape {image.shape}")
    if image.ndim == 3 and image.shape[-1] in (3, 4) and image.shape[0] > 4:
        raise ValueError(f"{path}: looks like an RGB image {image.shape}; the model expects grayscale")
    if image.dtype != np.uint16:
        warnings.warn(f"{path.name}: dtype is {image.dtype}, but the model was trained on 16-bit "
                      "images. Values are used as they are, which may give poor results.")


def segment(image, session, info, batch_size=16):
    """Label image (values 1-7) for a 2D slice or a 3D stack."""
    stack = image[None] if image.ndim == 2 else image
    labels = np.stack([predict_slice(s, session, info, batch_size).argmax(-1).astype(np.uint8) + 1
                       for s in stack])
    return labels[0] if image.ndim == 2 else labels


def natural_key(path):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", Path(path).name)]


def list_inputs(path):
    path = Path(path)
    if path.is_file():
        return [path]
    if not path.is_dir():
        raise FileNotFoundError(f"Input not found: {path}")
    files = sorted((p for p in path.iterdir() if p.suffix.lower() in TIFF_SUFFIXES), key=natural_key)
    if not files:
        raise FileNotFoundError(f"No .tif/.tiff files found in {path}")
    return files


def middle_slice(a):
    return a if a.ndim == 2 else a[a.shape[0] // 2]


def show_labels(ax, labels, classes, title):
    ax.imshow(labels, cmap=ListedColormap(LABEL_COLOURS[:len(classes)]),
              vmin=1, vmax=len(classes), interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")

## Load the model

In [ ]:
session, info = load_model(MODEL_PATH)
classes = info["classes"]

unknown = [c for c in SAVE_CLASSES if c not in classes]
if unknown:
    raise ValueError(f"Unknown class(es) in SAVE_CLASSES: {unknown}. Available: {classes}")

print(f"Model: {Path(MODEL_PATH).name}")
print(f"Tile size {info['tile']}, overlap {info['overlap']}, input scaled to 0-1: {info['scale_to_unit']}")
print(f"Classes (label 1-{len(classes)}): {classes}")

## Run the model

In [ ]:
files = list_inputs(INPUT_PATH)
out_dir = Path(OUTPUT_DIR)
(out_dir / "labels").mkdir(parents=True, exist_ok=True)
group_name = Path(INPUT_PATH).stem

results = {}   # file name -> (image, labels), used for the preview and validation
for f in files:
    image = tifffile.imread(f)
    check_image(image, f)
    labels = segment(image, session, info, BATCH_SIZE)

    tifffile.imwrite(out_dir / "labels" / f"{f.stem}.tif", labels, compression="zlib")
    for name in SAVE_CLASSES:
        folder = out_dir / f"{group_name}_{name}"
        folder.mkdir(exist_ok=True)
        mask = (labels == classes.index(name) + 1).astype(np.uint8) * 255
        tifffile.imwrite(folder / f"{f.stem}.tif", mask, compression="zlib")

    results[f.name] = (image, labels)
    print(f"{f.name}: shape {image.shape} done")

print(f"\nSaved to {out_dir.resolve()}")

## Preview
Shows the first input (middle slice for stacks). Change `PREVIEW_INDEX` to see another file.

In [ ]:
PREVIEW_INDEX = 0

name = list(results)[PREVIEW_INDEX]
image, labels = results[name]
fig, axes = plt.subplots(1, 2, figsize=(10, 6))
axes[0].imshow(middle_slice(image), cmap="gray")
axes[0].set_title(f"Input: {name}")
axes[0].axis("off")
show_labels(axes[1], middle_slice(labels), classes, "Prediction")
fig.legend(handles=[Patch(color=c, label=f"{i + 1} {n}") for i, (c, n) in enumerate(zip(LABEL_COLOURS, classes))],
           loc="center right")
plt.tight_layout(rect=(0, 0, 0.85, 1))
plt.show()

## Validation (optional)

Compares the Python prediction with a segmentation exported from arivis/ZEN for the
same image. Skipped when `VALIDATION_PATH = None`.

- **Pixel agreement**: fraction of pixels with the same label (label images only).
- **Dice** per class: 1 = identical, 0 = no overlap.
- **Difference map**: pixels where the two disagree.

In [ ]:
def base_name(path):
    """File name without extension(s) and without known extra endings."""
    name = Path(path).name
    for ext in (".ome.tiff", ".ome.tif", ".tiff", ".tif"):
        if name.lower().endswith(ext):
            name = name[:-len(ext)]
            break
    return re.sub(r"(_finalprediction(_ome)?|_raw)$", "", name, flags=re.IGNORECASE)


def find_reference(input_file):
    ref = Path(VALIDATION_PATH)
    if ref.is_file():
        if len(files) > 1:
            raise ValueError("VALIDATION_PATH is a single file but INPUT_PATH has several files; "
                             "use a folder of references")
        return ref
    matches = [p for p in ref.iterdir()
               if p.suffix.lower() in TIFF_SUFFIXES and base_name(p) == base_name(input_file)]
    if len(matches) > 1:
        raise ValueError(f"{input_file.name}: several possible references: {[m.name for m in matches]}")
    return matches[0] if matches else None


def read_reference(path, n_classes, pred_ndim):
    """Read a reference segmentation as a label image (values 1..n_classes).

    Accepts a label image, or the arivis/ZEN export with one 0/255 channel per class
    (e.g. *_finalprediction.ome.tiff, shape (n_classes, rows, cols)).
    """
    ref = tifffile.imread(path)
    if ref.ndim == pred_ndim + 1:
        axes = [i for i, s in enumerate(ref.shape) if s == n_classes]
        if not axes:
            raise ValueError(f"{path}: cannot find a class axis of size {n_classes} in shape {ref.shape}")
        return np.moveaxis(ref, axes[0], 0).argmax(0).astype(np.int64) + 1
    return ref.astype(np.int64)


def dice(a, b):
    denom = a.sum() + b.sum()
    return 2 * (a & b).sum() / denom if denom else float("nan")


if VALIDATION_PATH is None:
    print("Validation skipped (VALIDATION_PATH = None)")
else:
    for f in files:
        ref_path = find_reference(f)
        if ref_path is None:
            print(f"{f.name}: no reference with the same name found, skipped")
            continue
        _, pred = results[f.name]
        if VALIDATION_CLASS is None:
            ref = read_reference(ref_path, len(classes), pred.ndim)
        else:
            ref = tifffile.imread(ref_path)
        if ref.shape != pred.shape:
            print(f"{f.name}: shape mismatch (prediction {pred.shape}, reference {ref.shape}), skipped")
            continue

        print(f"\n{f.name}  vs  {ref_path.name}")
        if VALIDATION_CLASS is None:
            if tifffile.imread(ref_path).ndim == pred.ndim:                  # label image
                ref = ref - REFERENCE_FIRST_LABEL + 1                          # convert to labels 1-7
            values = np.unique(ref)
            if values.min() < 1 or values.max() > len(classes):
                warnings.warn(f"Reference values {values.tolist()} are outside 1-{len(classes)} after "
                              "conversion; check REFERENCE_FIRST_LABEL and VALIDATION_CLASS")
            print(f"  Pixel agreement: {(pred == ref).mean():.4f}")
            for label, cname in enumerate(classes, start=1):
                d = dice(pred == label, ref == label)
                print(f"  Dice {cname:<11} " + ("not present in either image" if np.isnan(d) else f"{d:.4f}"))
            disagree = pred != ref
        else:
            label = classes.index(VALIDATION_CLASS) + 1
            p, r = pred == label, ref > 0
            print(f"  Dice {VALIDATION_CLASS}: {dice(p, r):.4f}")
            disagree = p != r

        fig, axes = plt.subplots(1, 3, figsize=(14, 6))
        if VALIDATION_CLASS is None:
            show_labels(axes[0], middle_slice(pred), classes, "Python prediction")
            show_labels(axes[1], middle_slice(ref), classes, "arivis/ZEN reference")
        else:
            axes[0].imshow(middle_slice(p), cmap="gray"); axes[0].set_title(f"Python: {VALIDATION_CLASS}")
            axes[1].imshow(middle_slice(r), cmap="gray"); axes[1].set_title(f"Reference: {VALIDATION_CLASS}")
        axes[2].imshow(middle_slice(disagree), cmap="magma")
        axes[2].set_title(f"Disagreement ({middle_slice(disagree).mean():.2%} of pixels)")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()